### School Data Merge notebook

##### Once the team has been able to merge their files across all years for their respective categories, we need to create an overall dataset with all of these features

### This script provides a basic template for how we could merge all these categories into one unified dataset:
- Merges through combokey
- Keeps identifying information that may be found across those categories such as (State, etc.)
- Schools must be present across all years

## For example:

| YEAR | COMBOKEY | SCH_NAME | ... | SCHOOL_SUPPORT_F1 | REFARREST_F1 | ... |
| --- | --- | --- | --- | --- | --- | --- |
| 2017-2018 | 1345 | Arizona High School | ... | 45 | 90 | ... | 
| 2020-2021| 1345 | Arizona High School | ... | 35 | 170 | ... | 

- Data for the school for one year == the row
- Columns are the features

In [13]:
from pathlib import Path
import re

import pandas as pd



# Map each category to its merged CSV file.
CATEGORY_FILES = {
    "referrals_arrests": Path("../data/cleaned/merged_refarrests.csv"),
    "school_support": Path("../data/cleaned/merged_schoolsupport.csv"),
    "offenses": Path("../data/cleaned/merged_offenses.csv"),
    "covid": Path("../data/cleaned/merged_covid_indicators.csv"),
    "enrollment": Path("../data/cleaned/merged_enrollment.csv"),
    "expulsions": Path("../data/cleaned/merged_expulsions.csv"),
    "harrasement_bullying": Path("../data/cleaned/merged_harrasementbullying.csv"),
    "justice_facilities": Path("../data/cleaned/merged_justicefacilities.csv"),
    "suspensions": Path("../data/cleaned/merged_suspensions.csv"),
    # "restraint_seclusion": Path("../data/cleaned/merged_restraint_seclusion.csv"),
    # "enrollment": Path("../data/cleaned/merged_enrollment.csv"),
}

# Each row represents one school in one year.
KEYS = ["YEAR", "COMBOKEY"]

# These columns describe the school and should agree wherever they appear.
IDENTIFIERS = [
    "LEA_STATE",
    "LEA_STATE_NAME",
    "LEAID",
    "LEA_NAME",
    "SCHID",
    "SCH_NAME",
    "SCH_JUST",
]

OUTPUT_FILE = Path("../data/cleaned/unified_school_year.csv")


def load_category(category: str, path: Path) -> pd.DataFrame:
    """Load and validate one prepared category file."""
    if not path.exists():
        raise FileNotFoundError(f"{category} file not found: {path}")

    # Read IDs as strings so values such as COMBOKEY aren't treated as numbers.
    string_columns = KEYS + IDENTIFIERS + ["SCHOOL_YEAR", "JJ"]
    df = pd.read_csv(
        path,
        dtype={column: "string" for column in string_columns},
    )

    # Standardize category files that call the year column SCHOOL_YEAR.
    if "YEAR" not in df.columns and "SCHOOL_YEAR" in df.columns:
        df = df.rename(columns={"SCHOOL_YEAR": "YEAR"})

    # JJ and SCH_JUST both identify juvenile justice facilities.
    if "JJ" in df.columns:
        if "SCH_JUST" in df.columns:
            jj = df["JJ"].str.strip().str.casefold()
            sch_just = df["SCH_JUST"].str.strip().str.casefold()
            mismatch = jj.notna() & sch_just.notna() & jj.ne(sch_just)

            if mismatch.any():
                examples = df.loc[mismatch, KEYS + ["JJ", "SCH_JUST"]].head(10)
                raise ValueError(
                    f"{category} has mismatched JJ and SCH_JUST values:\n{examples}"
                )

            df["SCH_JUST"] = df["SCH_JUST"].fillna(df["JJ"])
            df = df.drop(columns="JJ")
        else:
            df = df.rename(columns={"JJ": "SCH_JUST"})

    missing_keys = set(KEYS) - set(df.columns)
    if missing_keys:
        raise ValueError(
            f"{category} is missing required key columns: "
            f"{sorted(missing_keys)}"
        )

    # Trim whitespace from keys and identifiers.
    for column in KEYS + IDENTIFIERS:
        if column in df.columns:
            df[column] = df[column].str.strip()

    if df[KEYS].isna().any().any() or df[KEYS].eq("").any().any():
        raise ValueError(f"{category} contains missing or blank merge keys")

    def normalize_year(year: str) -> str:
        match = re.fullmatch(r"(\d{4})-(\d{2}|\d{4})", year)
        if not match:
            return year

        start_year = int(match.group(1))
        end_text = match.group(2)
        end_year = int(end_text)
        if len(end_text) == 2:
            end_year += (start_year // 100) * 100
            if end_year < start_year:
                end_year += 100

        return f"{start_year:04d}-{end_year:04d}"

    df["YEAR"] = df["YEAR"].map(normalize_year)

    # A school can appear in multiple years, but not more than once per year.
    if df.duplicated(KEYS).any():
        examples = df.loc[df.duplicated(KEYS, keep=False), KEYS].head()
        raise ValueError(
            f"{category} has multiple rows for a school-year:\n{examples}"
        )

    return df


def normalized_for_comparison(values: pd.Series) -> pd.Series:
    """Ignore capitalization and repeated whitespace when comparing identifiers."""
    return (
        values.str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .str.casefold()
    )


if not CATEGORY_FILES:
    raise ValueError("Add at least one category file to CATEGORY_FILES")


frames = []
seen_features = set()
covid_frame = None

for category, path in CATEGORY_FILES.items():
    df = load_category(category, path)

    # Feature columns should be uniquely named across categories.
    features = set(df.columns) - set(KEYS) - set(IDENTIFIERS)
    overlapping_features = features & seen_features
    if overlapping_features:
        raise ValueError(
            f"{category} repeats feature names from an earlier category: "
            f"{sorted(overlapping_features)}"
        )

    seen_features.update(features)
    if category == "covid":
        covid_frame = df
    else:
        frames.append((category, df))

if not frames:
    raise ValueError("Add at least one non-COVID category to CATEGORY_FILES")

# COVID years are optional and must not affect the required year set.
all_years = set().union(*(set(df["YEAR"]) for _, df in frames))

# ------------
# Filter each category independently before merging categories.
filtered_frames = []

for category, df in frames:
    category_years = set(df["YEAR"].unique())
    years_per_school = df.groupby("COMBOKEY")["YEAR"].nunique()
    complete_schools = years_per_school[
        years_per_school == len(category_years)
    ].index

    filtered_df = df[df["COMBOKEY"].isin(complete_schools)].copy()

    if filtered_df.empty:
        raise ValueError(
            f"{category} has no schools present in every year: "
            f"{sorted(category_years)}"
        )

    print(
        f"{category}: retained {filtered_df['COMBOKEY'].nunique():,} of "
        f"{df['COMBOKEY'].nunique():,} schools across {len(category_years)} years"
    )
    filtered_frames.append((category, filtered_df))

frames = filtered_frames
if covid_frame is not None:
    frames.append(("covid", covid_frame))
# -------------- 
# Inner-join required categories; COVID is left-joined because its years
# may be missing for otherwise eligible schools.
unified = frames[0][1].copy()
schools_with_name_mismatches = set()

for category, df in frames[1:]:
    suffix = f"__{category}"

    unified = unified.merge(
        df,
        on=KEYS,
        how="left" if category == "covid" else "inner",
        validate="one_to_one",
        suffixes=("", suffix),
    )

    # If both files contain the same identifier, verify the values agree.
    for column in IDENTIFIERS:
        other_column = f"{column}{suffix}"

        if column in unified.columns and other_column in unified.columns:
            left = normalized_for_comparison(unified[column])
            right = normalized_for_comparison(unified[other_column])

            mismatch = (
                left.notna()
                & right.notna()
                & left.ne(right).fillna(False)
            )

            if mismatch.any():
                examples = unified.loc[
                    mismatch, KEYS + [column, other_column]
                ].head(10)

                if column == "SCHID":
                    print(
                        f"{category} has differing SCHID values; keeping "
                        f"the values in {column} and {other_column}:\n"
                        f"{examples.to_string(index=False)}"
                    )
                    continue
                elif column == "LEA_NAME":
                    schools_with_name_mismatches.update(
                        unified.loc[mismatch, "COMBOKEY"].dropna().unique()
                    )
                    print(
                        f"{category} has differing LEA_NAME values; "
                        f"those schools will be removed:\n"
                        f"{examples.to_string(index=False)}"
                    )
                else:
                    print(examples.to_string(index=False))
                    raise ValueError(
                        f"{category} has mismatched {column} values "
                        f"for matching school-years:\n{examples}"
                    )

            # Keep one copy after confirming the values agree.
            unified[column] = unified[column].fillna(unified[other_column])
            unified = unified.drop(columns=other_column)

    # Keep identifiers supplied only by this category.
    for column in IDENTIFIERS:
        other_column = f"{column}{suffix}"
        if other_column in unified.columns and column not in unified.columns:
            unified = unified.rename(columns={other_column: column})


if schools_with_name_mismatches:
    unified = unified[
        ~unified["COMBOKEY"].isin(schools_with_name_mismatches)
    ].copy()
    print(
        f"Removed {len(schools_with_name_mismatches):,} schools due to "
        "LEA_NAME mismatches across categories."
    )


# Keep only schools with a row for every year represented in the input files.
years_per_school = unified.groupby("COMBOKEY")["YEAR"].nunique()
complete_schools = years_per_school[
    years_per_school == len(all_years)
].index
unified = unified[unified["COMBOKEY"].isin(complete_schools)].copy()

if unified.empty:
    raise ValueError(
        "No schools have data in every category for every year represented "
        f"in the input files: {sorted(all_years)}"
    )

# Save the combined school-year dataset.
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
unified.to_csv(OUTPUT_FILE, index=False)

print(f"Saved {len(unified):,} school-year rows to {OUTPUT_FILE}")

referrals_arrests: retained 2,893 of 2,893 schools across 4 years
school_support: retained 7,212 of 7,212 schools across 4 years
offenses: retained 22,974 of 30,849 schools across 4 years
enrollment: retained 22,883 of 31,005 schools across 4 years
expulsions: retained 18,782 of 32,772 schools across 4 years
harrasement_bullying: retained 22,974 of 30,849 schools across 4 years
justice_facilities: retained 16,163 of 27,416 schools across 4 years
suspensions: retained 18,782 of 32,772 schools across 4 years
school_support has differing LEA_NAME values; those schools will be removed:
     YEAR     COMBOKEY                                           LEA_NAME                                                   LEA_NAME__school_support
2023-2024 262460006111              ÃÂ Morley Stanwood Community Schools                  ������������������������Morley Stanwood Community Schools
2017-2018 24SOP0299985 Maryland State Department of Education-Juvenile Se Maryland State Department of Education-